# FreshKart: Customer Support with LCEL

FreshKart gets thousands of tickets a day. The old bot had five flaws: **A1** no memory, **A2** invented data, **A3** rules hidden in the prompt, **A4** slow sequential calls, **A5** no streaming. We fix them one LangChain (LCEL) piece at a time and end with this pipeline:

```
 {input, chat_history}
        |
        v
 RunnableLambda(find_order_id)        regex: FK1001 / fk1001 / ord_1001
        |
        v
 RunnablePassthrough.assign(signals = RunnableParallel(
        intent (LLM) | order | customer | inventory | rider | safety | claimed_amount ))
        |
        v
 RunnableLambda(decide_route)         plain Python rules, no LLM
        |
        v
 RunnableBranch: auto_refund | wismo | substitution | info | policy_decline | human_review
        |
        v
 StrOutputParser -> streamed reply     (whole chain wrapped in RunnableWithMessageHistory)
```

## 1. Setup
**Why:** the bot must answer from *real* data (fixes A2: invented data), so we load FreshKart's CSV files once and keep them as pandas tables. The LLM is reached through OpenRouter; the key lives in `.env`, never in the notebook.

In [1]:
import os, re, time
import pandas as pd
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import (RunnableLambda, RunnablePassthrough, RunnableParallel,
                                      RunnableBranch)
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.chat_history import InMemoryChatMessageHistory

load_dotenv()

def get_llm():
    return ChatOpenAI(model=os.getenv("MODEL", "openai/gpt-4o-mini"),
                      base_url="https://openrouter.ai/api/v1",
                      api_key=os.getenv("OPENROUTER_API_KEY"), temperature=0)

def read(name):
    return pd.read_csv(f"data/{name}.csv", dtype=str, keep_default_na=False)

llm = get_llm()
orders, customers, inventory, riders, tickets = [read(n) for n in
                                                 ["orders", "customers", "inventory", "riders", "tickets"]]
print(len(orders), "orders,", len(customers), "customers,", len(inventory), "inventory rows,",
      len(riders), "riders,", len(tickets), "tickets")
TICKETS = tickets.to_dict("records")


30 orders, 12 customers, 25 inventory rows, 12 riders, 18 tickets


## 2. The chat model
**Why:** everything starts with one call to the LLM. A chat model takes a *list of messages* (a system message sets the behaviour, a human message is the customer). `.invoke` sends them and returns an `AIMessage`; the text is in `.content`.

In [2]:
reply = llm.invoke([SystemMessage(content="You are a polite FreshKart support agent. Reply in one short line."),
                    HumanMessage(content="Mera order late hai")])
print(reply.content)

Kya aap apna order number share kar sakte hain, taaki hum madad kar sakein?


## 3. Prompt templates
**Why:** the old bot glued strings together by hand. A `ChatPromptTemplate` keeps the instructions fixed and leaves `{holes}` for the customer's message, so the same prompt can be reused for every ticket (a first step against A3: instructions live in one place).

In [3]:
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a polite FreshKart support agent. Reply in one short line."),
    ("human", "{input}"),
])
print(prompt.invoke({"input": "Mera order late hai"}).to_messages())

[SystemMessage(content='You are a polite FreshKart support agent. Reply in one short line.', additional_kwargs={}, response_metadata={}), HumanMessage(content='Mera order late hai', additional_kwargs={}, response_metadata={})]


## 4. Chains: prompt | llm | parser (invoke, batch, stream)
**Why:** the `|` operator joins steps into one chain. Every chain gets `.invoke` (one input), `.batch` (many inputs at once, fixing A4: slowness) and `.stream` (tokens as they arrive, fixing A5: no streaming). `StrOutputParser` turns the AIMessage into a plain string.

In [4]:
chain = prompt | llm | StrOutputParser()
print(chain.invoke({"input": "Mera order late hai"}))
print(chain.batch([{"input": "Bread missing hai"}, {"input": "Refund kab milega?"}]))
for chunk in chain.stream({"input": "Delivery hours kya hain?"}):
    print(chunk, end="|")

Kya aap apne order ka order number share kar sakte hain, taaki hum madad kar sakein?
['Kya aap apne order details share kar sakte hain, taaki hum madad kar sakein?', 'Aapka refund 5-7 business dinon mein process hoga.']
Fresh|Kart| ki| delivery| hours| sub|ah| |9| b|aje| se| r|aat| |10| b|aje| tak| hain|.||||

## 5. RunnableLambda: any Python function becomes a chain step
**Why:** the order id is a fixed pattern (`FK1001`, `fk1001`, `ord_1001`). An LLM may guess a wrong id (A2), but a regex never invents one. `RunnableLambda` wraps the function so it works with `|`, `.invoke` and `.batch`.

In [5]:
def find_order_id(text):
    m = re.search(r"(?i)\b(?:FK|ord_)\s?(\d{4})\b", text)
    return "FK" + m.group(1) if m else None

order_id_step = RunnableLambda(find_order_id)
msgs = list(tickets.set_index("id").loc[["T06", "T10"], "message"])
print(order_id_step.invoke(msgs[0]))
print(order_id_step.batch(msgs))

FK1006
['FK1006', 'FK1010']


## 6. Data lookups and regex signals
**Why:** facts such as order status, refunds in the last 30 days or expiry dates must come from the CSVs, not from the model (A2). Each lookup is a small function that returns a plain dict, or `None` when nothing is found. Two regexes also read the message itself: a food-safety check and the amount the customer claims.

In [6]:
def row(df, col, val):
    m = df[df[col] == val]
    return m.iloc[0].to_dict() if len(m) else None

def get_order(order_id): return row(orders, "order_id", order_id)

def get_customer(order_id):
    o = get_order(order_id)
    return row(customers, "customer_id", o["customer_id"]) if o else None

def check_inventory(order_id):
    o = get_order(order_id)
    if not o: return None
    d = {"expired_confirmed": False, "baby": False}
    for item in o["items"].split(";"):
        r = row(inventory[inventory.dark_store == o["dark_store"]], "name", item)
        if r:
            d["expired_confirmed"] |= r["expiry_date"] < o["delivered_at"][:10]
            d["baby"] |= r["category"] == "baby_care"
    return d

def get_rider(order_id): return row(riders, "order_id", order_id)

SAFETY = (r"(?i)\b(?:food poisoning|vomit\w*|ulti|diarrhoea|diarrhea|allerg\w*|rash|illness|sick|"
          r"stomach pain|fever|hospital|infection|swelling|choking|fungus|mould|mold)\b")
def has_safety(text): return bool(re.search(SAFETY, text))

def claimed_amount(text):
    nums = re.findall(r"(?i)(?:₹|Rs\.?|INR)\s?(\d[\d,]*(?:\.\d+)?)", text)
    return max(float(n.replace(",", "")) for n in nums) if nums else None

print(get_order("FK1003")["items"], get_customer("FK1011"), check_inventory("FK1004"), get_rider("FK1014"))
print(has_safety("beti ko ulti ho gayi"), claimed_amount("₹520 twice debited"), claimed_amount("Rs. 1,200 aur ₹300"))

Mother Dairy Curd 400g;Tomato 1kg {'customer_id': 'C06', 'name': 'Vikram Rao', 'city': 'Bengaluru', 'tier': 'regular', 'pass_member': 'n', 'refunds_30d': '3', 'fraud_score': '0.85'} {'expired_confirmed': True, 'baby': True} {'rider_id': 'R09', 'order_id': 'FK1014', 'gps_delivered': 'n', 'complaints_30d': '0'}
True 520.0 1200.0


## 7. RunnablePassthrough.assign: add a key, keep the rest
**Why:** the chain carries one dict that grows step by step. `assign` computes a new key and keeps every old key, so the customer's `input` (and later `chat_history`) is never lost. This is also where memory (A1) plugs in: if the customer gave no order id now, we look back through earlier messages.

In [7]:
def order_id_from(x):
    for text in [x["input"]] + [m.content for m in reversed(x.get("chat_history", []))]:
        if find_order_id(text): return find_order_id(text)

with_id = RunnablePassthrough.assign(order_id=RunnableLambda(order_id_from))
print(with_id.invoke({"input": "fk1006 kab tak aayega?"}))

{'input': 'fk1006 kab tak aayega?', 'order_id': 'FK1006'}


## 8. RunnableParallel: all signals at the same time
**Why:** the old bot did one step after another (A4). The seven signals do not depend on each other, so `RunnableParallel` runs them together. Only `intent` calls the LLM: it returns `intent,sentiment`, and a small lambda parses it. Anything unknown becomes `unclear`, which later goes to a human.

In [8]:
INTENTS = """wismo: where is my order, delivery status, delay
missing_item: an item is not in the bag
damaged_spoiled: damaged, broken, leaking or spoiled item
expired: item is past its expiry date
wrong_item_substitution: wrong item or unwanted substitution / different brand
quality_complaint: poor quality (limp veg, stale) but not clearly spoiled
refund_status: where is my refund
payment_issue: double debit, payment failed but money cut
coupon_pass: coupon, FreshKart Pass, offers
food_safety: illness, allergy, contamination, foreign object
rider_misconduct: rude rider, demanded cash, harassment
general: anything else, FAQs"""

intent_prompt = ChatPromptTemplate.from_messages([
    ("system", "Classify the customer message into exactly one intent:\n" + INTENTS +
     "\nAlso give the sentiment: angry, neutral or happy.\nAnswer ONLY as: intent,sentiment"),
    ("human", "{input}"),
])

def parse_intent(text):
    label = text.strip().lower().split(",")[0].strip()
    return label if label in [l.split(":")[0] for l in INTENTS.split("\n")] else "unclear"

oid = lambda f: RunnableLambda(lambda x: f(x["order_id"]))
signals = RunnableParallel(
    intent=intent_prompt | llm | StrOutputParser() | RunnableLambda(parse_intent),
    order=oid(get_order), customer=oid(get_customer), inventory=oid(check_inventory), rider=oid(get_rider),
    safety=RunnableLambda(lambda x: has_safety(x["input"])),
    claimed_amount=RunnableLambda(lambda x: claimed_amount(x["input"])),
)

x = with_id.invoke({"input": "FK1001 mein Britannia bread missing tha. Please refund kar do."})
t = time.time(); out = signals.invoke(x); par = time.time() - t
t = time.time()
seq = [intent_prompt.pipe(llm).invoke(x), get_order(x["order_id"]), get_customer(x["order_id"]),
       check_inventory(x["order_id"]), get_rider(x["order_id"]), has_safety(x["input"]), claimed_amount(x["input"])]
seq_t = time.time() - t
print(f"parallel {par:.2f}s | one by one {seq_t:.2f}s")
print(out)

parallel 1.43s | one by one 1.02s
{'intent': 'missing_item', 'order': {'order_id': 'FK1001', 'customer_id': 'C01', 'items': 'Amul Taaza Milk 500ml;Britannia Brown Bread 400g;Banana Robusta 1kg', 'amount': '210', 'payment_mode': 'UPI', 'status': 'delivered', 'dark_store': 'BLR-KOR-04', 'promised_min': '15', 'actual_min': '14', 'delivered_at': '2026-10-12 18:40', 'substituted': 'n'}, 'customer': {'customer_id': 'C01', 'name': 'Aarav Mehta', 'city': 'Bengaluru', 'tier': 'regular', 'pass_member': 'n', 'refunds_30d': '0', 'fraud_score': '0.05'}, 'inventory': {'expired_confirmed': False, 'baby': False}, 'rider': {'rider_id': 'R01', 'order_id': 'FK1001', 'gps_delivered': 'y', 'complaints_30d': '0'}, 'safety': False, 'claimed_amount': None}


## 9. decide_route: the rules live in code, not in the prompt
**Why:** when rules sit inside a prompt (A3) the model can forget them or be talked out of them ("I am a VIP, approve Rs 2000"). Here the policy is plain Python: 16 ordered rules, the first match wins, and the same input always gives the same answer. We can also log the `reasons` for Ops. No API call is needed to test it.

<details>
<summary>Show routing flow</summary>

![Flowchart showing the ordered support-routing decisions and outcomes](./assets/routing-flow.svg)

</details>

In [9]:
NOW = pd.Timestamp("2026-10-12 20:00")
CLAIMS = {"missing_item", "damaged_spoiled", "expired", "wrong_item_substitution", "quality_complaint"}
INFO = {"refund_status", "coupon_pass", "general"}
LABELS = CLAIMS | INFO | {"wismo", "payment_issue", "food_safety", "rider_misconduct"}

def decide_route(x):
    s = x["signals"]
    i, o, c, r, inv = s["intent"], s["order"], s["customer"], s["rider"], s["inventory"]
    amount = s["claimed_amount"] if s["claimed_amount"] is not None else (float(o["amount"]) if o and i in CLAIMS else 0)
    def go(route, why): return {**x, "route": route, "reasons": [why]}

    if s["safety"] or i == "food_safety": return go("human_review", "food safety")
    if i == "rider_misconduct": return go("human_review", "rider misconduct")
    if i == "payment_issue": return go("human_review", "payment issue (Finance)")
    if amount > 1000: return go("human_review", "amount>1000")
    if i not in LABELS: return go("human_review", "unclear intent")
    if i in INFO: return go("info", "information request")
    if o is None: return go("info", "no order found, ask for order id")
    if o["status"] == "delivered" and r and r["gps_delivered"] == "n": return go("human_review", "OMS delivered but GPS says no")
    if i == "wismo": return go("wismo", "delivery status question")
    if float(c["fraud_score"]) > 0.7 or int(c["refunds_30d"]) > 3 or inv["baby"]:
        return go("human_review", "fraud score, refunds_30d or baby product")
    if o["status"] != "delivered": return go("policy_decline", "order not delivered")
    if (NOW - pd.Timestamp(o["delivered_at"])) > pd.Timedelta(hours=48): return go("policy_decline", "outside 48 hours")
    if i == "expired" and not inv["expired_confirmed"]: return go("policy_decline", "expiry not confirmed")
    if i == "wrong_item_substitution" and o["substituted"] == "y": return go("substitution", "substituted order")
    if amount <= 300: return go("auto_refund", "amount<=300, clean customer")
    return go("human_review", "amount between 301 and 1000")

clean = {"customer_id": "C01", "fraud_score": "0.05", "refunds_30d": "0"}
base = {"intent": "missing_item", "order": {"amount": "210", "status": "delivered", "delivered_at": "2026-10-12 18:40", "substituted": "n"},
        "customer": clean, "inventory": {"expired_confirmed": False, "baby": False},
        "rider": {"gps_delivered": "y"}, "safety": False, "claimed_amount": None}
for name, change in [("small clean claim", {}), ("VIP asks Rs 2000", {"claimed_amount": 2000.0}),
                     ("baby product", {"inventory": {"expired_confirmed": False, "baby": True}})]:
    print(name, "->", decide_route({"signals": {**base, **change}})["route"])

small clean claim -> auto_refund
VIP asks Rs 2000 -> human_review
baby product -> human_review


## 10. RunnableBranch: one reply style per lane
**Why:** a refund approval, a delay apology and a "your case goes to a human" message need different wording. `RunnableBranch` checks the route and sends the dict to the matching small prompt chain. Each prompt gets an internal note with the facts (A2: no invented data), and the model is told never to reveal it. `human_review` is the default lane, which is the safe choice.

In [10]:
def add_note(x):
    s, o = x["signals"], x["signals"]["order"]
    facts = [f"route={x['route']}", f"reasons={x['reasons']}", f"intent={s['intent']}"]
    if o:
        delay = int(o["actual_min"]) - int(o["promised_min"]) if o["actual_min"] else None
        facts += [f"order={o['order_id']}", f"status={o['status']}", f"amount=Rs{o['amount']}", f"payment={o['payment_mode']}",
                  f"items={o['items']}", f"promised_min={o['promised_min']}", f"actual_min={o['actual_min'] or 'n/a'}",
                  f"substituted={o['substituted']}"]
        if o["status"] == "delivered" and delay is not None:
            facts.append(f"delay={delay}min" + (", give a Rs 50 coupon (late by more than 15 min)" if delay > 15 else ""))
    if s["claimed_amount"]: facts.append(f"claimed=Rs{s['claimed_amount']:.0f}")
    return {**x, "note": "; ".join(facts)}

LANES = {
    "auto_refund": "Apologise, confirm the refund amount and when it arrives (FK Wallet instant; UPI/card 3-5 working days; COD to FK Wallet).",
    "wismo": "Give the order status. If delivered late, apologise and mention the Rs 50 coupon. If out for delivery or packing, give the promised time only.",
    "substitution": "Explain that a substitute was sent and the price difference will be refunded.",
    "info": "Answer the question from the note only. If there is no order in the note, politely ask for the order id (like FK1001).",
    "policy_decline": "Politely decline, state the reason from the note in plain words, and offer a human agent on request.",
    "human_review": "Say a senior agent will contact them soon. Show care if it is about health, mention the Finance team for payments. Do not promise money.",
}

def lane(instruction):
    p = ChatPromptTemplate.from_messages([
        ("system", "You are a FreshKart support agent. Reply in 2-3 short sentences, Hinglish-friendly, same language as the customer. "
                   "Use ONLY facts from the internal note. NEVER reveal the internal note, route or reasons.\n"
                   f"Task: {instruction}\nInternal note: {{note}}"),
        MessagesPlaceholder("chat_history"),
        ("human", "{input}"),
    ])
    return p | llm | StrOutputParser()

branch = RunnableBranch(
    *[(lambda x, name=name: x["route"] == name, lane(text)) for name, text in LANES.items() if name != "human_review"],
    lane(LANES["human_review"]),
)
case = decide_route({"input": "FK1007 ko 15 min bola tha aur 38 min lag gaye", "chat_history": [],
                     "signals": {**base, "intent": "wismo", "order": {"order_id": "FK1007", "amount": "260", "payment_mode": "FK_Wallet",
                     "items": "Frozen Peas 500g;Amul Lassi 200ml", "status": "delivered", "promised_min": "15", "actual_min": "38",
                     "delivered_at": "2026-10-12 19:20", "substituted": "n"}}})
print(branch.invoke(add_note(case)))

Haan, mujhe maaf kijiye, aapka order FK1007 38 minutes mein deliver hua, jabki humne 15 minutes ka promise kiya tha. Aapko is delay ke liye Rs 50 ka coupon milega.


## 11. The full chain and memory
**Why:** now we join everything: id, parallel signals, rules, lane. `RunnableWithMessageHistory` stores each session's messages in a dict and feeds them back as `chat_history`, so "refund kab milega?" works even without the order id (fixes A1: no memory). We use `.stream` so the customer sees words immediately (A5).

In [11]:
full_chain = (RunnablePassthrough.assign(order_id=RunnableLambda(order_id_from))
              | RunnablePassthrough.assign(signals=signals)
              | RunnableLambda(decide_route)
              | RunnableLambda(add_note)
              | branch)

store = {}
def get_history(session_id):
    return store.setdefault(session_id, InMemoryChatMessageHistory())

bot = RunnableWithMessageHistory(full_chain, get_history, input_messages_key="input", history_messages_key="chat_history")
cfg = {"configurable": {"session_id": "demo"}}

for msg in ["FK1001 mein Britannia bread missing tha, refund do", "refund kab milega?"]:
    print("Customer:", msg, "\nBot: ", end="")
    for chunk in bot.stream({"input": msg}, config=cfg):
        print(chunk, end="")
    print("\n")

Customer: FK1001 mein Britannia bread missing tha, refund do 
Bot: 

/Users/yash/Documents/FDE/CodeRepo/freshkart/.venv/lib/python3.12/site-packages/IPython/core/interactiveshell.py:3823: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)
/var/folders/2v/3ksq7t197913_mwh94x6sjbc0000gn/T/ipykernel_62901/199454230.py:9: LangChainDeprecationWarning: The class `InMemoryChatMessageHistory` was deprecated in LangChain 1.6.4 and will be removed in 2.0.0 See the short-term memory documentation for recommended alternatives: https://docs.langchain.com/oss/python/langchain/short-term-memory
  return store.setdefault(session_id, InMemoryChatMessageHistory())


Haan, mujhe maaf kijiye ki aapka Britannia Brown Bread missing tha. Aapka Rs. 210 ka refund process ho gaya hai, jo UPI payment ke liye 3-5 working days mein aapke account mein aa jayega. Agar aapko aur koi madad chahiye, toh zaroor batayein!

Customer: refund kab milega? 
Bot: Aapka Rs. 210 ka refund 3-5 working days mein aapke account mein aa jayega. Agar aapko aur koi sawaal hai, toh pooch sakte hain!



## 12. Evaluation: do the rules route all 18 tickets correctly?
**Why:** a rule-based pipeline can be *tested*. We run only the routing part (signals + `decide_route`, no reply) over every ticket with `.batch` and compare with the expected route.

In [12]:
router = (RunnablePassthrough.assign(order_id=RunnableLambda(order_id_from))
          | RunnablePassthrough.assign(signals=signals) | RunnableLambda(decide_route))
results = router.batch([{"input": m, "chat_history": []} for m in tickets["message"]], config={"max_concurrency": 6})
df = tickets[["id", "expected_route"]].copy()
df["intent"] = [r["signals"]["intent"] for r in results]
df["route"] = [r["route"] for r in results]
df["ok"] = ["✅" if a == b else "❌" for a, b in zip(df.expected_route, df.route)]
print(df.to_string(index=False))
print("Score:", (df.ok == "✅").sum(), "/", len(df))

 id expected_route                  intent          route ok
T01    auto_refund            missing_item    auto_refund  ✅
T02   substitution wrong_item_substitution   substitution  ✅
T03    auto_refund                 expired    auto_refund  ✅
T04   human_review                 expired   human_review  ✅
T05 policy_decline                 expired policy_decline  ✅
T06          wismo                   wismo          wismo  ✅
T07          wismo                   wismo          wismo  ✅
T08   human_review         damaged_spoiled   human_review  ✅
T09   human_review             food_safety   human_review  ✅
T10   human_review        rider_misconduct   human_review  ✅
T11   human_review            missing_item   human_review  ✅
T12   human_review            missing_item   human_review  ✅
T13   human_review           payment_issue   human_review  ✅
T14   human_review                   wismo   human_review  ✅
T15           info                   wismo           info  ✅
T16           info      

## 13. Exercise 1: PII Shield Before the LLM
**Why:** DPDP Act compliance (case study §7). Customer PII (phone, email, UPI, card numbers) must not reach LLM prompts. We add a station at the front of the belt using `RunnablePassthrough.assign(masked_input=..., pii_count=...)`.
- The intent prompt and reply lanes read `masked_input`.
- Regex signals (`order_id`, `claimed_amount`, `safety`) read the raw `input`.
- Order IDs (e.g. `FK1001`) and rupee amounts (e.g. `₹520`) are never masked.

In [13]:
# Cell 13a: Regex PII Masking
def mask_pii(text):
    count = 0
    # 1. Card: 12-16 digits with optional spaces or hyphens
    def sub_card(m):
        nonlocal count; count += 1; return "[CARD]"
    text = re.sub(r"\b(?:\d[ -]?){11,15}\d\b", sub_card, text)

    # 2. Email: domain with dot extension
    def sub_email(m):
        nonlocal count; count += 1; return "[EMAIL]"
    text = re.sub(r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b", sub_email, text)

    # 3. UPI id: handle@bank
    def sub_upi(m):
        nonlocal count; count += 1; return "[UPI]"
    text = re.sub(r"\b[a-zA-Z0-9.\-_]{2,}@[a-zA-Z]{2,}\b", sub_upi, text)

    # 4. Phone: 10-digit Indian mobile starting with 6-9, optional +91
    def sub_phone(m):
        nonlocal count; count += 1; return "[PHONE]"
    text = re.sub(r"\b(?:\+?91[ -]?)?[6-9]\d{4}[ -]?\d{5}\b", sub_phone, text)

    return text, count

# Unit test on sample message
sample_pii = "Please call me on 98450 12345 or mail ravi.k@gmail.com, UPI ravi@okaxis. FK1001 mein Britannia bread missing tha, refund do."
masked, pii_count = mask_pii(sample_pii)
print("Original:", sample_pii)
print("Masked:  ", masked)
print("Count:   ", pii_count)

Original: Please call me on 98450 12345 or mail ravi.k@gmail.com, UPI ravi@okaxis. FK1001 mein Britannia bread missing tha, refund do.
Masked:   Please call me on [PHONE] or mail [EMAIL], UPI [UPI]. FK1001 mein Britannia bread missing tha, refund do.
Count:    3


In [14]:
# Cell 13b: Shielded Pipeline Assembly
# Intent prompt gets an adapter to read masked_input
signals_shielded = RunnableParallel(
    intent=RunnableLambda(lambda x: {"input": x["masked_input"]}) | intent_prompt | llm | StrOutputParser() | RunnableLambda(parse_intent),
    order=oid(get_order), customer=oid(get_customer), inventory=oid(check_inventory), rider=oid(get_rider),
    safety=RunnableLambda(lambda x: has_safety(x["input"])),
    claimed_amount=RunnableLambda(lambda x: claimed_amount(x["input"])),
)

# router masks text at station 1, then attaches order_id, signals, route
router_shielded = (RunnablePassthrough.assign(
                       masked_input=RunnableLambda(lambda x: mask_pii(x["input"])[0]),
                       pii_count=RunnableLambda(lambda x: mask_pii(x["input"])[1]))
                   | RunnablePassthrough.assign(order_id=RunnableLambda(order_id_from))
                   | RunnablePassthrough.assign(signals=signals_shielded)
                   | RunnableLambda(decide_route))

# Lanes receive masked_input
def add_note_shielded(x):
    base = add_note(x)
    return {**base, "input": x.get("masked_input", x["input"])}

reply_chain_shielded = RunnableLambda(add_note_shielded) | branch
full_chain_shielded = router_shielded | reply_chain_shielded

In [15]:
# Cell 13c: Acceptance Tests & Verification
# Test 1: Verify masked string seen by intent prompt
prompt_output = intent_prompt.invoke({"input": mask_pii(sample_pii)[0]}).to_string()
print("Intent prompt receives:\n", prompt_output.split("\n")[-1])

# Test 2: Verify router output for sample ticket with PII
res = router_shielded.invoke({"input": sample_pii, "chat_history": []})
print("\nSample routing result:")
print("Route:     ", res["route"])
print("Order ID:  ", res["order_id"])
print("PII Masked:", res["pii_count"])

Intent prompt receives:
 Human: Please call me on [PHONE] or mail [EMAIL], UPI [UPI]. FK1001 mein Britannia bread missing tha, refund do.

Sample routing result:
Route:      auto_refund
Order ID:   FK1001
PII Masked: 3


## 14. Exercise 2: Language Signal + Multilingual Reply (Combined 1-call Optimization)
**Why:** FreshKart receives tickets in English, Hinglish, Hindi, Kannada, and Telugu (case study §3). Ops needs language visibility in the queue, and the customer must receive replies in their language.
**Design:** We optimize by combining `intent`, `sentiment`, and `language` into a single LLM prompt (`nlu_prompt`). This halves API token costs and request latency while keeping `signals = RunnableParallel` clean.

In [16]:
# Cell 14a: Combined NLU (Intent + Sentiment + Language in 1 call)
LANGUAGES = {"english", "hinglish", "hindi", "kannada", "telugu", "other"}
VALID_INTENTS = [l.split(":")[0] for l in INTENTS.split("\n")]

nlu_prompt = ChatPromptTemplate.from_messages([
    ("system", "Classify customer text into:\n"
               "1. Intent:\n" + INTENTS + "\n"
               "2. Sentiment: angry, neutral, happy\n"
               "3. Language: english, hinglish (Hindi words written using Latin/English alphabet, or code-mixed Hindi-English), hindi (ONLY if written in Devanagari script like हिन्दी), kannada, telugu, other\n"
               "Answer ONLY as: intent,sentiment,language"),
    ("human", "{input}"),
])

def parse_nlu(text, model_name="main"):
    parts = [p.strip().lower() for p in text.strip().split(",")]
    intent = parts[0] if len(parts) > 0 and parts[0] in VALID_INTENTS else "unclear"
    sentiment = parts[1] if len(parts) > 1 and parts[1] in {"angry", "neutral", "happy"} else "neutral"
    language = parts[2] if len(parts) > 2 and parts[2] in LANGUAGES else "english"
    return {"intent": intent, "sentiment": sentiment, "language": language, "model_used": model_name}

nlu_chain = RunnableLambda(lambda x: {"input": x["masked_input"]}) | nlu_prompt | llm | StrOutputParser() | RunnableLambda(parse_nlu)


In [17]:
# Cell 14b: Multilingual Pipeline Assembly
signals_v2 = RunnableParallel(
    intent=RunnableLambda(lambda x: x["nlu"]["intent"]),
    language=RunnableLambda(lambda x: x["nlu"]["language"]),
    sentiment=RunnableLambda(lambda x: x["nlu"]["sentiment"]),
    order=oid(get_order), customer=oid(get_customer), inventory=oid(check_inventory), rider=oid(get_rider),
    safety=RunnableLambda(lambda x: has_safety(x["input"])),
    claimed_amount=RunnableLambda(lambda x: claimed_amount(x["input"])),
)

router_v2 = (RunnablePassthrough.assign(
                 masked_input=RunnableLambda(lambda x: mask_pii(x["input"])[0]),
                 pii_count=RunnableLambda(lambda x: mask_pii(x["input"])[1]))
             | RunnablePassthrough.assign(order_id=RunnableLambda(order_id_from))
             | RunnablePassthrough.assign(nlu=nlu_chain)
             | RunnablePassthrough.assign(signals=signals_v2)
             | RunnableLambda(decide_route))

def add_note_v2(x):
    s, o = x["signals"], x["signals"]["order"]
    lang = s.get("language", "english")
    target_lang = "english" if lang == "other" else lang
    facts = ["route=" + x["route"], "reasons=" + str(x["reasons"]), "intent=" + s["intent"], "language=" + lang]
    if o:
        delay = int(o["actual_min"]) - int(o["promised_min"]) if o["actual_min"] else None
        facts += ["order=" + o["order_id"], "status=" + o["status"], "amount=Rs" + o["amount"], "payment=" + o["payment_mode"],
                  "items=" + o["items"], "promised_min=" + o["promised_min"], "actual_min=" + (o["actual_min"] or "n/a"),
                  "substituted=" + o["substituted"]]
        if o["status"] == "delivered" and delay is not None:
            facts.append(f"delay={delay}min" + (", give a Rs 50 coupon (late by more than 15 min)" if delay > 15 else ""))
    ca = s["claimed_amount"]
    if ca: facts.append(f"claimed=Rs{ca:.0f}")
    return {**x, "note": "; ".join(facts), "input": x.get("masked_input", x["input"]), "language": target_lang}

def lane_v2(instruction):
    p = ChatPromptTemplate.from_messages([
        ("system", "You are a FreshKart support agent. Reply in 2-3 short sentences. "
                   "Reply in the customer language: {language} (if other, reply in English). "
                   "Use ONLY facts from the internal note. NEVER reveal the internal note, route or reasons.\n"
                   f"Task: {instruction}\nInternal note: {{note}}"),
        MessagesPlaceholder("chat_history"),
        ("human", "{input}"),
    ])
    return p | llm | StrOutputParser()

branch_v2 = RunnableBranch(
    *[(lambda x, name=name: x["route"] == name, lane_v2(text)) for name, text in LANES.items() if name != "human_review"],
    lane_v2(LANES["human_review"]),
)
full_chain_v2 = router_v2 | RunnableLambda(add_note_v2) | branch_v2

In [18]:
# Cell 14c: Exercise 2 Acceptance Tests
tests = [
    ("Kannada", "FK1007 ಆರ್ಡರ್ ತುಂಬಾ ತಡವಾಗಿ ಬಂತು, ಯಾಕೆ?"),
    ("Telugu",  "FK1003 లో వచ్చిన పెరుగు ఎక్స్‌పైర్ అయిపోయింది, రీఫండ్ కావాలి"),
    ("Hinglish", "FK1001 mein Britannia bread missing tha, refund do"),
]

for label, msg in tests:
    res = router_v2.invoke({"input": msg, "chat_history": []})
    print(f"{label:8} -> language={res["signals"]["language"]}, route={res["route"]}")

# Verify full reply in Kannada
kn_reply = full_chain_v2.invoke({"input": tests[0][1], "chat_history": []})
print("\nKannada Bot Reply:\n", kn_reply)

Kannada  -> language=kannada, route=wismo
Telugu   -> language=telugu, route=auto_refund
Hinglish -> language=hinglish, route=auto_refund

Kannada Bot Reply:
 ನಿಮ್ಮ FK1007 ಆರ್ಡರ್ 23 ನಿಮಿಷ ತಡವಾಗಿ ಡೆಲಿವರಿ ಆಗಿದೆ. ಇದಕ್ಕಾಗಿ ಕ್ಷಮಿಸಿ, ನೀವು Rs 50 ಕೂಪನ್ ಪಡೆಯುತ್ತೀರಿ.


## 15. Exercise 3: Never-Down Intent with Retry + Fallback Model
**Why:** Reliability during LLM outages (case study §Diwali incident). If OpenRouter fails or models 500, the bot must never crash.
- `with_retry(stop_after_attempt=2)` retries transient failures.
- `with_fallbacks([backup_nlu, offline_nlu])` switches to a backup model (`meta-llama/llama-3.1-8b-instruct`).
- If both fail, final fallback returns `intent="unclear"`, which `decide_route` safely routes to `human_review`.
- Reply lanes fall back to a polite apology without raising exceptions.

In [19]:
# Cell 15a: Fallback Model & Chain Setup
def get_backup_llm():
    return ChatOpenAI(model=os.getenv("BACKUP_MODEL", "meta-llama/llama-3.1-8b-instruct"),
                      base_url="https://openrouter.ai/api/v1",
                      api_key=os.getenv("OPENROUTER_API_KEY"), temperature=0, max_retries=0)

backup_llm = get_backup_llm()

def parse_nlu(text, model_name="main"):
    parts = [p.strip().lower() for p in text.strip().split(",")]
    intent = parts[0] if len(parts) > 0 and parts[0] in VALID_INTENTS else "unclear"
    sentiment = parts[1] if len(parts) > 1 and parts[1] in {"angry", "neutral", "happy"} else "neutral"
    language = parts[2] if len(parts) > 2 and parts[2] in LANGUAGES else "english"
    return {"intent": intent, "sentiment": sentiment, "language": language, "model_used": model_name}

primary_nlu = (nlu_prompt | llm | StrOutputParser() | RunnableLambda(lambda t: parse_nlu(t, "main"))).with_retry(stop_after_attempt=2)
backup_nlu = (nlu_prompt | backup_llm | StrOutputParser() | RunnableLambda(lambda t: parse_nlu(t, "backup"))).with_retry(stop_after_attempt=2)
offline_nlu = RunnableLambda(lambda _: {"intent": "unclear", "sentiment": "neutral", "language": "english", "model_used": "offline_fallback"})

robust_nlu_chain = RunnableLambda(lambda x: {"input": x["masked_input"]}) | primary_nlu.with_fallbacks([backup_nlu, offline_nlu])


In [20]:
# Cell 15b: Fault-tolerant Router & Lanes
signals_v3 = RunnableParallel(
    intent=RunnableLambda(lambda x: x["nlu"]["intent"]),
    language=RunnableLambda(lambda x: x["nlu"]["language"]),
    sentiment=RunnableLambda(lambda x: x["nlu"]["sentiment"]),
    model_used=RunnableLambda(lambda x: x["nlu"]["model_used"]),
    order=oid(get_order), customer=oid(get_customer), inventory=oid(check_inventory), rider=oid(get_rider),
    safety=RunnableLambda(lambda x: has_safety(x["input"])),
    claimed_amount=RunnableLambda(lambda x: claimed_amount(x["input"])),
)

router_v3 = (RunnablePassthrough.assign(
                 masked_input=RunnableLambda(lambda x: mask_pii(x["input"])[0]),
                 pii_count=RunnableLambda(lambda x: mask_pii(x["input"])[1]))
             | RunnablePassthrough.assign(order_id=RunnableLambda(order_id_from))
             | RunnablePassthrough.assign(nlu=robust_nlu_chain)
             | RunnablePassthrough.assign(signals=signals_v3)
             | RunnableLambda(decide_route))

def add_note_v3(x):
    s, o = x["signals"], x["signals"]["order"]
    lang = s.get("language", "english")
    target_lang = "english" if lang == "other" else lang
    facts = ["route=" + x["route"], "reasons=" + str(x["reasons"]), "intent=" + s["intent"], "language=" + lang]
    if o:
        delay = int(o["actual_min"]) - int(o["promised_min"]) if o["actual_min"] else None
        facts += ["order=" + o["order_id"], "status=" + o["status"], "amount=Rs" + o["amount"], "payment=" + o["payment_mode"],
                  "items=" + o["items"], "promised_min=" + o["promised_min"], "actual_min=" + (o["actual_min"] or "n/a"),
                  "substituted=" + o["substituted"]]
        if o["status"] == "delivered" and delay is not None:
            facts.append(f"delay={delay}min" + (", give a Rs 50 coupon (late by more than 15 min)" if delay > 15 else ""))
    ca = s["claimed_amount"]
    if ca: facts.append(f"claimed=Rs{ca:.0f}")
    return {**x, "note": "; ".join(facts), "input": x.get("masked_input", x["input"]), "language": target_lang}

FALLBACK_REPLY = "Sorry, we are facing a technical issue. A FreshKart agent will contact you shortly."

def safe_lane(instruction):
    p = ChatPromptTemplate.from_messages([
        ("system", "You are a FreshKart support agent. Reply in 2-3 short sentences. "
                   "Reply in the customer language: {language} (if other, reply in English). "
                   "Use ONLY facts from the internal note. NEVER reveal the internal note, route or reasons.\n"
                   f"Task: {instruction}\nInternal note: {{note}}"),
        MessagesPlaceholder("chat_history"),
        ("human", "{input}"),
    ])
    primary = (p | llm | StrOutputParser()).with_retry(stop_after_attempt=2)
    backup = (p | backup_llm | StrOutputParser()).with_retry(stop_after_attempt=2)
    fallback = RunnableLambda(lambda _: FALLBACK_REPLY)
    return primary.with_fallbacks([backup, fallback])

branch_v3 = RunnableBranch(
    *[(lambda x, name=name: x["route"] == name, safe_lane(text)) for name, text in LANES.items() if name != "human_review"],
    safe_lane(LANES["human_review"]),
)
full_chain_v3 = router_v3 | RunnableLambda(add_note_v3) | branch_v3


In [21]:
# Cell 15c: Exercise 3 Acceptance Tests
TICKETS = tickets.to_dict("records")
# 1. Normal execution
res = router_v3.batch([{"input": t["message"], "chat_history": []} for t in TICKETS])
score = sum(r["route"] == t["expected_route"] for r, t in zip(res, TICKETS))
m_used = res[0]["signals"]["model_used"]
print(f"Normal key score: {score} / 18 | Model: {m_used}")

# 2. Total outage simulation: offline fallback routes to human_review
broken_nlu = RunnableLambda(lambda _: {"intent": "unclear", "sentiment": "neutral", "language": "english", "model_used": "offline_fallback"})
router_offline = (RunnablePassthrough.assign(
                      masked_input=RunnableLambda(lambda x: mask_pii(x["input"])[0]),
                      pii_count=RunnableLambda(lambda x: mask_pii(x["input"])[1]))
                  | RunnablePassthrough.assign(order_id=RunnableLambda(order_id_from))
                  | RunnablePassthrough.assign(nlu=broken_nlu)
                  | RunnablePassthrough.assign(signals=signals_v3)
                  | RunnableLambda(decide_route))
off_res = router_offline.invoke({"input": "FK1001 missing bread", "chat_history": []})
off_route = off_res["route"]
off_model = off_res["signals"]["model_used"]
print(f"Offline outage route: {off_route} | Model: {off_model}")


Normal key score: 18 / 18 | Model: main
Offline outage route: human_review | Model: offline_fallback


## 16. Exercise 4: Ops Hand-off Card for Escalations
**Why:** For `human_review` escalations, senior agents take 2-3 minutes reading the raw conversation. We generate an internal **Ops hand-off card** in parallel with the customer reply:
```json
{"team": "food_safety", "priority": "P1", "summary_en": "...", "next_step": "..."}
```
- For `human_review` tickets: `reply_chain` yields both customer reply and structured card via `RunnableParallel(reply=branch, card=card_branch)`.
- For all other routes (`auto_refund`, `wismo`, etc.): `card = None` with 0 additional LLM calls.
- Design: placing `card_branch` in parallel after `branch` cleanly isolates escalation card generation without touching the 5 standard response lanes.

In [22]:
# Cell 16a: Ops Hand-off Card Chain
from langchain_core.output_parsers import JsonOutputParser

card_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an internal operations triage assistant for FreshKart.\n"
               "For this escalated customer ticket, create an Ops hand-off card as a JSON object.\n"
               "Rules:\n"
               "- team must be exactly one of: food_safety, finance, rider_ops, support\n"
               "  (use food_safety for illness/vomiting/allergy, finance for double debit/wallet, rider_ops for misconduct, support for other)\n"
               "- priority must be exactly one of: P1, P2, P3\n"
               "  (use P1 for food poisoning/health/safety, P2 for money/fraud/misconduct, P3 for general escalations)\n"
               "- summary_en: a concise 1-sentence summary in English\n"
               "- next_step: concrete immediate ops action (e.g., call customer, file FSSAI report, initiate bank trace)\n"
               "Internal note: {note}\n"
               "Answer ONLY valid JSON with keys: team, priority, summary_en, next_step."),
    ("human", "{input}"),
])

TEAMS = {"food_safety", "finance", "rider_ops", "support"}
PRIORITIES = {"P1", "P2", "P3"}

def validate_card(data):
    if not isinstance(data, dict): return None
    team = str(data.get("team", "")).lower().strip()
    priority = str(data.get("priority", "")).upper().strip()
    return {
        "team": team if team in TEAMS else "support",
        "priority": priority if priority in PRIORITIES else "P2",
        "summary_en": data.get("summary_en", "Customer escalation"),
        "next_step": data.get("next_step", "Contact customer"),
    }

card_chain = (card_prompt | llm | JsonOutputParser() | RunnableLambda(validate_card)).with_fallbacks([RunnableLambda(lambda _: None)])
card_branch = RunnableBranch(
    (lambda x: x["route"] == "human_review", card_chain),
    RunnableLambda(lambda _: None)
)

In [23]:
# Cell 16b: Parallel Reply & Card Belt Assembly
# reply_chain now emits a dict: {"reply": ..., "card": ...}
reply_chain_v4 = RunnableLambda(add_note_v3) | RunnableParallel(reply=branch_v3, card=card_branch)
full_chain_v4 = router_v3 | reply_chain_v4

In [24]:
# Cell 16c: Exercise 4 Acceptance Tests
test_cases = ["T09", "T13", "T10", "T01"]
for tid in test_cases:
    t = next(x for x in TICKETS if x["id"] == tid)
    res = full_chain_v4.invoke({"input": t["message"], "chat_history": []})
    print(f"--- {tid} ---")
    print("Card:", res["card"])
    print("Reply preview:", res["reply"][:60] + "...")


--- T09 ---
Card: {'team': 'food_safety', 'priority': 'P1', 'summary_en': "Customer's daughter is experiencing vomiting and stomach pain after consuming Chicken Biryani from order FK1009, indicating possible food poisoning.", 'next_step': 'call customer'}
Reply preview: Humein aapki chinta hai aur aapki beti ki sehat sabse pehle ...
--- T13 ---
Card: {'team': 'finance', 'priority': 'P2', 'summary_en': 'Customer reported double debit of ₹520 for order FK1013.', 'next_step': 'initiate bank trace'}
Reply preview: Aapki samasya ke liye humein khed hai. Ek senior agent aapse...
--- T10 ---
Card: {'team': 'rider_ops', 'priority': 'P2', 'summary_en': 'Customer reported rude behavior and misconduct from the rider during delivery.', 'next_step': 'call customer'}
Reply preview: Humein aapki samasya ke liye khed hai. Ek senior agent aapse...
--- T01 ---
Card: None
Reply preview: Haan, mujhe aapki takleef ke liye maaf kijiye. Aapka Rs. 210...


## 17. Exercise 5: Bulk Triage for the Morning Backlog
**Why:** Morning backlog has ~1,200 tickets overnight. Ops needs them routed rapidly before 9 am without generating replies.
- `router.batch(inputs, config={"max_concurrency": 5}, return_exceptions=True)`
- `return_exceptions=True`: individual failing rows yield `route="error"` without crashing the entire batch.
- `max_concurrency=5`: protects against API provider rate limits (HTTP 429) while delivering $\ge 2\times$ throughput over sequential processing.

In [25]:
# Cell 17a: Bulk Triage Runnable
def run_bulk_triage(tickets_list, concurrency=5):
    inputs = [{"input": str(t.get("message", "")), "chat_history": []} for t in tickets_list]
    raw_results = router_v3.batch(inputs, config={"max_concurrency": concurrency}, return_exceptions=True)
    output = []
    for t, res in zip(tickets_list, raw_results):
        if isinstance(res, Exception):
            output.append({"id": t.get("id"), "order_id": None, "intent": "error", "route": "error", "reasons": str(res)})
        else:
            output.append({
                "id": t.get("id"), "order_id": res.get("order_id"),
                "intent": res["signals"]["intent"], "route": res["route"],
                "reasons": "; ".join(res.get("reasons", []))
            })
    return output

bulk_triage = RunnableLambda(run_bulk_triage)

In [26]:
# Cell 17b: Accuracy & Resiliency Tests
import time

# Test 1: Accuracy on data/tickets.csv
t0 = time.time()
triaged = bulk_triage.invoke(TICKETS)
batch_duration = time.time() - t0
score = sum(r["route"] == t["expected_route"] for r, t in zip(triaged, TICKETS))
print(f"Accuracy on {len(TICKETS)} tickets: {score} / {len(TICKETS)} in {batch_duration:.2f}s")

# Test 2: Resiliency on empty / broken row
resilient_test = [{"id": "T_EMPTY", "message": ""}, TICKETS[0]]
res_out = bulk_triage.invoke(resilient_test)
print("\nResiliency Test Rows:")
for r in res_out:
    rid, rroute, rreasons = r["id"], r["route"], r["reasons"]
    print(f"  {rid}: route={rroute}, reasons={rreasons}")


Accuracy on 18 tickets: 18 / 18 in 6.15s

Resiliency Test Rows:
  T_EMPTY: route=info, reasons=information request
  T01: route=auto_refund, reasons=amount<=300, clean customer


In [27]:
# Cell 17c: Concurrency Benchmark (Sequential vs Batch)
benchmark_sample = TICKETS[:5]

# Sequential loop
t0 = time.time()
seq_res = [router_v3.invoke({"input": t["message"], "chat_history": []}) for t in benchmark_sample]
seq_duration = time.time() - t0

# Batch with concurrency=5
t0 = time.time()
batch_res = router_v3.batch([{"input": t["message"], "chat_history": []} for t in benchmark_sample], config={"max_concurrency": 5})
batch_duration = time.time() - t0

speedup = seq_duration / batch_duration if batch_duration > 0 else 0
print(f"Sequential (5 items): {seq_duration:.2f}s")
print(f"Batch (concurrency 5): {batch_duration:.2f}s")
print(f"Speedup: {speedup:.1f}x (Acceptance requirement: >= 2.0x)")


Sequential (5 items): 5.64s
Batch (concurrency 5): 1.73s
Speedup: 3.3x (Acceptance requirement: >= 2.0x)


## 18. Exercise 6: Model Switcher at Runtime
**Why:** CTO question: "Is `gpt-4o-mini` really the best value? Try Llama or stronger models." Rebuilding chains per model is error-prone.
- LCEL `ConfigurableField(id="model")` + `.configurable_alternatives()` configures the base model once.
- Request config `{"configurable": {"model": "llama"}}` dynamically routes to Llama without altering the belt.
- Invariant: T18 (VIP ₹2000) routes to `human_review` under **every** model because money policy is deterministic Python in `decide_route`, never left to LLM whim.

In [28]:
# Cell 18a: Configurable Alternatives on Base LLM
from langchain_core.runnables import ConfigurableField

mini_llm = ChatOpenAI(model="openai/gpt-4o-mini", base_url="https://openrouter.ai/api/v1", api_key=os.getenv("OPENROUTER_API_KEY"), temperature=0, max_retries=0)
llama_llm = ChatOpenAI(model="meta-llama/llama-3.1-8b-instruct", base_url="https://openrouter.ai/api/v1", api_key=os.getenv("OPENROUTER_API_KEY"), temperature=0, max_retries=0)

llm_switchable = mini_llm.configurable_alternatives(
    ConfigurableField(id="model"),
    default_key="mini",
    llama=llama_llm,
)

# Re-wrap primary NLU using switchable LLM
primary_nlu_switchable = (nlu_prompt | llm_switchable | StrOutputParser() | RunnableLambda(lambda t: parse_nlu(t, "main"))).with_retry(stop_after_attempt=2)
robust_nlu_switchable = RunnableLambda(lambda x: {"input": x["masked_input"]}) | primary_nlu_switchable.with_fallbacks([backup_nlu, offline_nlu])

router_v6 = (RunnablePassthrough.assign(
                 masked_input=RunnableLambda(lambda x: mask_pii(x["input"])[0]),
                 pii_count=RunnableLambda(lambda x: mask_pii(x["input"])[1]))
             | RunnablePassthrough.assign(order_id=RunnableLambda(order_id_from))
             | RunnablePassthrough.assign(nlu=robust_nlu_switchable)
             | RunnablePassthrough.assign(signals=signals_v3)
             | RunnableLambda(decide_route))

In [29]:
# Cell 18b: Acceptance Tests - Compare Models & T18 Invariant
import time

def compare_models(model_keys=["mini", "llama"]):
    results = {}
    for m in model_keys:
        t0 = time.time()
        batch_res = router_v6.batch([{"input": t["message"], "chat_history": []} for t in TICKETS],
                                    config={"configurable": {"model": m}, "max_concurrency": 5})
        elapsed = time.time() - t0
        score = sum(r["route"] == t["expected_route"] for r, t in zip(batch_res, TICKETS))
        results[m] = {"score": score, "time": elapsed}
        print(f"Model \x27{m}\x27: {score} / 18 correct in {elapsed:.2f}s")
    return results

res_comparison = compare_models()

# Acceptance test 2: T18 routing check on both models
t18 = next(x for x in TICKETS if x["id"] == "T18")
r_mini = router_v6.invoke({"input": t18["message"], "chat_history": []}, config={"configurable": {"model": "mini"}})
r_llama = router_v6.invoke({"input": t18["message"], "chat_history": []}, config={"configurable": {"model": "llama"}})
print("\nT18 Route with mini :", r_mini["route"], "| Reasons:", r_mini["reasons"])
print("T18 Route with llama:", r_llama["route"], "| Reasons:", r_llama["reasons"])


Model 'mini': 18 / 18 correct in 4.95s
Model 'llama': 9 / 18 correct in 4.77s

T18 Route with mini : human_review | Reasons: ['amount>1000']
T18 Route with llama: human_review | Reasons: ['amount>1000']


## 19. Exercise 7: Live Pipeline Tracker (Listeners & Overlap)
**Why:** Ops does not trust a black box. As stations finish, we watch the belt and measure exact timings.

- **Station naming**: .

- **Capture mechanism**: .

  - *Why listeners over *:  runs synchronously without async loop conflicts in Streamlit, capturing microsecond wall times on worker threads.

  - Listeners append  into a thread-safe list in .

  - When  is omitted from , the listener returns immediately with 0 overhead (fully optional).

- **Parallel Overlap**: Stations inside  execute concurrently, so sum of station times > wall duration.

In [30]:
# Cell 19a: Station Tracker with .with_listeners
def make_tracker_listener():
    def on_end(run, config):
        tracker = config.get("configurable", {}).get("tracker")
        if tracker is not None:
            dur = (run.end_time - run.start_time).total_seconds() if (run.end_time and run.start_time) else 0.0
            tracker.append({
                "name": run.name,
                "start_time": run.start_time,
                "end_time": run.end_time,
                "duration": dur,
                "outputs": getattr(run, "outputs", None),
            })
    return on_end

tracker_listener = make_tracker_listener()

def track(r, name):
    return r.with_config(run_name=name).with_listeners(on_end=tracker_listener)

# Wrap stations with tracking
signals_v7 = RunnableParallel(
    intent=RunnableLambda(lambda x: x["nlu"]["intent"]).with_config(run_name="signal_intent"),
    language=RunnableLambda(lambda x: x["nlu"]["language"]).with_config(run_name="signal_language"),
    sentiment=RunnableLambda(lambda x: x["nlu"]["sentiment"]).with_config(run_name="signal_sentiment"),
    model_used=RunnableLambda(lambda x: x["nlu"]["model_used"]).with_config(run_name="signal_model_used"),
    order=track(oid(get_order), "order_lookup"),
    customer=track(oid(get_customer), "customer_lookup"),
    inventory=track(oid(check_inventory), "inventory_lookup"),
    rider=track(oid(get_rider), "rider_lookup"),
    safety=track(RunnableLambda(lambda x: has_safety(x["input"])), "safety_check"),
    claimed_amount=track(RunnableLambda(lambda x: claimed_amount(x["input"])), "amount_extract"),
)

router_v7 = (
    RunnablePassthrough.assign(
        masked_input=track(RunnableLambda(lambda x: mask_pii(x["input"])[0]), "pii_masking"),
        pii_count=RunnableLambda(lambda x: mask_pii(x["input"])[1])
    )
    | RunnablePassthrough.assign(order_id=track(RunnableLambda(order_id_from), "find_order_id"))
    | RunnablePassthrough.assign(nlu=track(robust_nlu_switchable, "intent_llm"))
    | RunnablePassthrough.assign(signals=track(signals_v7, "signals_parallel"))
    | track(RunnableLambda(decide_route), "decide_route")
)

reply_chain_v7 = (
    track(RunnableLambda(add_note), "add_note")
    | RunnableParallel(
        reply=track(branch, "reply_generation"),
        card=track(card_branch, "card_generation")
    )
)


In [31]:
# Cell 19b: Acceptance Tests - T09 Timings, Parallel Overlap & Optionality
t09 = next(t for t in TICKETS if t["id"] == "T09")

# Test 1: Tracked run on T09
events_t09 = []
t0 = time.time()
res_t09 = router_v7.invoke({"input": t09["message"], "chat_history": []}, config={"configurable": {"tracker": events_t09}})
router_time = time.time() - t0

reply_events_t09 = []
t0 = time.time()
reply_res_t09 = reply_chain_v7.invoke(res_t09, config={"configurable": {"tracker": reply_events_t09}})
reply_time = time.time() - t0

all_events = events_t09 + reply_events_t09
print(f"T09 Route: {res_t09['route']} (Router: {router_time:.2f}s, Reply: {reply_time:.2f}s)
Station Timings:")
for e in sorted(all_events, key=lambda x: x["start_time"]):
    print(f"  ✓ {e['name']:<20} {e['duration']:.4f}s")

# Test 2: Overlap verification in parallel step
parallel_event = next(e for e in events_t09 if e["name"] == "signals_parallel")
parallel_signals = [e for e in events_t09 if e["name"] in {"order_lookup", "customer_lookup", "inventory_lookup", "rider_lookup", "safety_check", "amount_extract"}]
sum_signals = sum(e["duration"] for e in parallel_signals)
print(f"
Parallel signals wall time: {parallel_event['duration']:.4f}s")
print(f"Sum of signal station times: {sum_signals:.4f}s")

reply_event = next(e for e in reply_events_t09 if e["name"] == "reply_generation")
card_event = next(e for e in reply_events_t09 if e["name"] == "card_generation")
sum_parallel_reply = reply_event["duration"] + card_event["duration"]
print(f"Reply + Card parallel sum:   {sum_parallel_reply:.4f}s vs Reply chain wall time: {reply_time:.4f}s")
assert sum_parallel_reply > reply_time * 0.8  # proves parallel execution savings!

# Test 3: Tracker is fully optional
res_no_track = router_v7.invoke({"input": t09["message"], "chat_history": []})
assert res_no_track["route"] == "human_review"
print("
Tracker turn-off test passed (0 overhead, optional)!")


## Recap: concept → what it fixed

| Concept | What it fixed |
|---|---|
| `ChatPromptTemplate` | Instructions in one reusable place instead of glued strings |
| `prompt \| llm \| StrOutputParser` | One simple chain with `.invoke`, `.batch`, `.stream` |
| `.batch` / `RunnableParallel` | A4: slow sequential calls, signals now run together |
| `.stream` | A5: no streaming, the reply appears word by word |
| `RunnableLambda` + pandas/regex | A2: invented data, facts come from the CSVs |
| `RunnablePassthrough.assign` | Keep the input and add new keys as the chain grows |
| `decide_route` | A3: rules in the prompt, now ordered, testable Python |
| `RunnableBranch` | A reply style per lane, safe `human_review` default |
| `RunnableWithMessageHistory` | A1: no memory, the order id is recovered from earlier turns |